In [1]:
import os
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
DATE_START = pd.Timestamp("2022-07-01 00:00:00")
DATE_END   = pd.Timestamp("2025-06-30 23:00:00")

# Imputed output directories from config.yaml
datasets = {
    "CPCB (India)":      "/home/rishi/ML Projects/Air Pollution/CPCB/sites_imputed",
    "EPA (USA)":         "/home/rishi/ML Projects/Air Pollution/EPA/imputed",
    "AURN (UK)":         "/home/rishi/ML Projects/Air Pollution/AURN/imputed",
    "EEA FR (France)":   "/home/rishi/ML Projects/Air Pollution/EEA/FR/imputed",
    "EEA DE (Germany)":  "/home/rishi/ML Projects/Air Pollution/EEA/DE/imputed",
    "SINAICA (Mexico)":  "/home/rishi/ML Projects/Air Pollution/SINAICA/imputed",
    "CNEMC (China)":     "/home/rishi/ML Projects/Air Pollution/CNEMC/imputed",
}

In [3]:
def _count_file(filepath, date_start, date_end):
    try:
        df = pd.read_csv(filepath, parse_dates=["Timestamp"])
    except (ValueError, KeyError):
        return 0, 0, None
    mask = (df["Timestamp"] >= date_start) & (df["Timestamp"] <= date_end)
    df = df.loc[mask]
    pollutant_col = df.columns[1]
    stem = filepath.stem
    site_id = "_".join(stem.rsplit("_", 1)[:-1])
    return len(df), int(df[pollutant_col].notna().sum()), site_id


def count_dataset(directory, date_start, date_end, max_workers=24):
    csv_files = sorted(Path(directory).glob("*.csv"))
    total = 0
    non_nan = 0
    sites = set()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_count_file, f, date_start, date_end): f for f in csv_files}
        for future in tqdm(as_completed(futures), total=len(futures), desc=Path(directory).name, leave=False):
            t, nn, site_id = future.result()
            total   += t
            non_nan += nn
            if site_id is not None:
                sites.add(site_id)

    return total, non_nan, len(sites)

In [4]:
rows = []
for name, directory in datasets.items():
    if not os.path.isdir(directory):
        print(f"[SKIP] Directory not found: {directory}")
        continue
    print(f"Processing {name} ...")
    total, non_nan, n_sites = count_dataset(directory, DATE_START, DATE_END)
    rows.append({
        "Dataset":          name,
        "Unique Sites":     n_sites,
        "Total Points":     total,
        "Non-NaN Points":   non_nan,
        "NaN Points":       total - non_nan,
        "Completeness (%)": round(100 * non_nan / total, 2) if total > 0 else 0.0,
    })
    print(f"  sites={n_sites:,}  total={total:,}  non-nan={non_nan:,}")

Processing CPCB (India) ...


sites_imputed:   0%|          | 0/1104 [00:00<?, ?it/s]

  sites=211  total=29,039,616  non-nan=29,039,616
Processing EPA (USA) ...


imputed:   0%|          | 0/1667 [00:00<?, ?it/s]

  sites=1,011  total=43,848,768  non-nan=43,848,768
Processing AURN (UK) ...


...

  sites=405  total=35,089,536  non-nan=35,089,536
Processing SINAICA (Mexico) ...


imputed:   0%|          | 0/75 [00:00<?, ?it/s]

  sites=24  total=1,972,800  non-nan=1,972,800
Processing CNEMC (China) ...


imputed:   0%|          | 0/8893 [00:00<?, ?it/s]

  sites=1,491  total=233,921,472  non-nan=233,921,472


In [5]:
df_results = (
    pd.DataFrame(rows)
    .sort_values("Dataset")
    .reset_index(drop=True)
)

df_display = df_results.copy()
for col in ["Unique Sites", "Total Points", "Non-NaN Points", "NaN Points"]:
    df_display[col] = df_display[col].apply(lambda x: f"{x:,}")

df_display.style.set_caption(f"Imputed data point counts per dataset  |  {DATE_START.date()} – {DATE_END.date()}")

,Dataset,Unique Sites,Total Points,Non-NaN Points,NaN Points,Completeness (%)
0,AURN (UK),113,"6,812,736","6,812,736",0,100.000000
1,CNEMC (China),"1,491","233,921,472","233,921,472",0,100.000000
2,CPCB (India),211,"29,039,616","29,039,616",0,100.000000
3,EEA DE (Germany),405,"35,089,536","35,089,536",0,100.000000
4,EEA FR (France),380,"21,227,328","21,227,328",0,100.000000
5,EPA (USA),"1,011","43,848,768","43,848,768",0,100.000000
6,SINAICA (Mexico),24,"1,972,800","1,972,800",0,100.000000


In [6]:
df_results.sum()

Dataset             AURN (UK)CNEMC (China)CPCB (India)EEA DE (Germ...
Unique Sites                                                     3635
Total Points                                                371912256
Non-NaN Points                                              371912256
NaN Points                                                          0
Completeness (%)                                                700.0
dtype: object